# 02 · Preprocesado

**Qué hace este notebook:** convierte la tabla de `data/silver` en la matriz de
modelado de `data/gold`, y guarda el transformador ajustado en `models/`. Es el
único sitio donde se decide qué se imputa, cómo se escala y qué variables
derivadas existen.

**La regla que lo gobierna todo:** *nada se ajusta con datos de test*. La
partición temporal se hace **antes** de imputar y escalar, y las medias,
desviaciones y medianas salen sólo de `train`. Hacerlo al revés es la fuga de
información más común y la más difícil de ver después: no rompe nada, sólo hace
que las métricas de `04` sean optimistas y que el modelo se degrade en
producción sin explicación.

**Entradas y salidas**

| | |
|---|---|
| Lee | `data/silver/<DATASET>`, y las decisiones escritas en `01_eda.ipynb` |
| Escribe | `data/gold/model_matrix.parquet`, `models/preprocessor.joblib` |
| Escribe | `reports/preprocessing/summary.csv` |

**Cuándo pasa a `src/`:** cuando `LAGS`, `DROP` y la estrategia de imputación
dejen de cambiar. Entonces esto se convierte en una etapa de
`src/packagename/etl/pipeline.py`, registrada después de las etapas de las que
depende, y `just etl` pasa a rehacerla con el resto del pipeline. Mientras siga
cambiando cada tarde, una etapa en `src/` sólo añade ceremonia.

## Parámetros

Estos valores son las conclusiones de `01` escritas en código. Cada entrada
debería poder señalar la sección del EDA que la justifica; el comentario está
para eso.

In [ ]:
DATASET = "measurements.parquet"
TARGET = "swh"
TIME = "time"
GROUP = None

OUTPUT = "model_matrix.parquet"

# Variables descartadas en 01: casi constantes, o con demasiados huecos. Anotar
# el motivo -- una lista de nombres sin razón no se puede revisar.
DROP: dict[str, str] = {
    # "algo": "casi constante (§1.1)",
}

# Retardos a construir, en pasos del índice, tal como salieron de §3.6 del EDA.
# Sólo retardos positivos: un valor futuro como predictor es fuga.
LAGS: dict[str, list[int]] = {
    # "msl": [3, 6, 12],
    # "u10": [3, 6],
}

# Ventanas para medias móviles, en pasos. Un estado del mar depende del viento
# acumulado, no del instantáneo, así que estas suelen ganar a los retardos
# sueltos.
ROLLING: dict[str, list[int]] = {
    # "u10": [6, 24],
}

# Variables direccionales, en grados. Se transforman a seno y coseno: 359° y 1°
# son direcciones casi idénticas y numéricamente opuestas, así que dejarlas en
# grados introduce una discontinuidad artificial que ningún modelo puede
# aprender.
CIRCULAR: list[str] = ["mwd"]

# Partición temporal por fechas, no por porcentajes: así las particiones no se
# mueven cuando llegan datos nuevos, y una métrica de hoy es comparable con la
# de la semana pasada.
TRAIN_END = "2015-12-31"
VALID_END = "2018-12-31"

# Hueco entre particiones, en pasos. Debe superar la longitud de decorrelación
# medida en §2.3 del EDA: sin él, las últimas horas de train y las primeras de
# validación son prácticamente la misma observación.
EMBARGO = 24

# Hueco máximo (en pasos) que se interpola. Por encima de esto la imputación
# dejaría de describir el dato y empezaría a inventarlo, así que la fila se cae.
MAX_INTERPOLATION = 3

# "standard" centra y escala; "robust" usa mediana e IQR, más apropiado cuando
# 01 encontró colas gruesas -- lo habitual en oleaje.
SCALER = "robust"

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

from packagename import get_settings, set_seed, setup_logging
from packagename.etl import read_table, write_table
from packagename.viz import COLOR_NAMES, apply_style, savefig

setup_logging(level="INFO")
apply_style("paper")

settings = get_settings()
set_seed(settings.random_seed)
settings.paths.ensure()

FIG = "preprocessing"
TABLES = settings.paths.reports / "preprocessing"
TABLES.mkdir(parents=True, exist_ok=True)

In [ ]:
raw = read_table(settings.paths.silver / DATASET)
raw[TIME] = pd.to_datetime(raw[TIME])
data = raw.set_index(TIME).sort_index()

step = data.index.to_series().diff().mode().iloc[0]
print(f"{len(data):,} filas, paso {step}")

## 1. Limpieza

Tres cosas, en este orden: quitar lo que `01` declaró inservible, imponer la
malla temporal regular, y convertir los valores físicamente imposibles en huecos.

El último punto es el que se olvida. Un `-999` o una altura de ola de 200 m no es
un outlier a decidir más tarde: es un valor que nunca existió, y dejarlo pasar
contamina la mediana con la que después se imputa todo lo demás.

In [ ]:
PHYSICAL_RANGES: dict[str, tuple[float, float]] = {
    "swh": (0.0, 30.0),
    "mwp": (0.0, 30.0),
    "mwd": (0.0, 360.0),
    "msl": (85000.0, 110000.0),
    "sst": (270.0, 320.0),
    "u10": (-60.0, 60.0),
    "v10": (-60.0, 60.0),
}

clean = data.drop(columns=list(DROP), errors="ignore")

# La malla regular hace que un retardo de k pasos sea siempre el mismo intervalo
# de tiempo. Sin ella, `shift(3)` cruza los huecos y relaciona instantes que
# pueden estar a semanas de distancia.
if GROUP is None:
    clean = clean.asfreq(step)

invalid = pd.Series(0, index=clean.columns, dtype=int)
for name, (low, high) in PHYSICAL_RANGES.items():
    if name in clean.columns:
        outside = (clean[name] < low) | (clean[name] > high)
        invalid[name] = int(outside.sum())
        clean.loc[outside, name] = np.nan

print(f"Descartadas: {list(DROP) or 'ninguna'}")
print(f"Valores anulados por rango físico: {int(invalid.sum())}")
invalid[invalid > 0]

## 2. Variables derivadas

Se construyen **antes** de partir, y eso es deliberado a pesar de lo que se dijo
arriba: un retardo o una media móvil se calculan sólo con el pasado de la propia
serie, así que no hay estadístico ajustado y no hay fuga. Lo que no puede
construirse antes de partir es cualquier cosa que mire *toda* la muestra: una
media global, un percentil, un escalado. Eso es la sección 4.

El precio de los retardos son las primeras filas, que quedan incompletas. Se
resuelve al final, cuando ya se sabe cuántas son.

In [ ]:
# -> src/packagename/features/build.py cuando LAGS y ROLLING dejen de cambiar.
def add_lags(frame: pd.DataFrame, lags: dict[str, list[int]]) -> pd.DataFrame:
    columns = {
        f"{name}_lag{lag}": frame[name].shift(lag)
        for name, steps in lags.items()
        if name in frame.columns
        for lag in steps
    }
    return frame.assign(**columns)


def add_rolling(frame: pd.DataFrame, windows: dict[str, list[int]]) -> pd.DataFrame:
    # `closed="left"` excluye el instante actual de la ventana, de modo que la
    # variable resume estrictamente el pasado. Sin ello la media móvil incluye el
    # propio t y la variable deja de ser una predictora honesta.
    columns = {
        f"{name}_mean{window}": frame[name].rolling(window, closed="left").mean()
        for name, sizes in windows.items()
        if name in frame.columns
        for window in sizes
    }
    return frame.assign(**columns)


def encode_circular(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    present = [name for name in columns if name in frame.columns]
    radians = {name: np.deg2rad(frame[name]) for name in present}
    encoded = {
        f"{name}_{part}": function(values)
        for name, values in radians.items()
        for part, function in (("sin", np.sin), ("cos", np.cos))
    }
    return frame.assign(**encoded).drop(columns=present)


def add_calendar(frame: pd.DataFrame) -> pd.DataFrame:
    # El día del año entra como par seno/coseno por la misma razón que la
    # dirección: el 31 de diciembre y el 1 de enero están juntos en el ciclo y
    # lejísimos en el entero.
    angle = 2 * np.pi * frame.index.dayofyear / 365.25
    return frame.assign(
        doy_sin=np.sin(angle),
        doy_cos=np.cos(angle),
        hour_sin=np.sin(2 * np.pi * frame.index.hour / 24),
        hour_cos=np.cos(2 * np.pi * frame.index.hour / 24),
    )


engineered = (
    clean.pipe(add_lags, LAGS)
    .pipe(add_rolling, ROLLING)
    .pipe(encode_circular, CIRCULAR)
    .pipe(add_calendar)
)

print(f"{clean.shape[1]} -> {engineered.shape[1]} columnas")
sorted(set(engineered.columns) - set(clean.columns))

## 3. Partición temporal

Por fechas y con embargo. El embargo es la parte que suele faltar: con datos
horarios y una ACF que tarda un día en caer, las últimas horas de train y las
primeras de validación son casi la misma observación, y el modelo obtiene crédito
por recordarla.

Para regionalización hay una segunda fuga posible, espacial: si el mismo temporal
aparece en un punto de train y en otro de test, dividir sólo por tiempo no basta.
Cuando `GROUP` deje de ser `None`, la partición tiene que ser por bloques de
tiempo **y** de espacio.

In [ ]:
# -> src/packagename/data/splits.py
def temporal_split(frame: pd.DataFrame, train_end: str, valid_end: str, embargo: int) -> pd.Series:
    gap = embargo * step
    train_cut = pd.Timestamp(train_end)
    valid_cut = pd.Timestamp(valid_end)

    labels = pd.Series("unused", index=frame.index, dtype="object")
    labels[frame.index <= train_cut] = "train"
    labels[(frame.index > train_cut + gap) & (frame.index <= valid_cut)] = "valid"
    labels[frame.index > valid_cut + gap] = "test"
    return labels


split = temporal_split(engineered, TRAIN_END, VALID_END, EMBARGO)
counts = split.value_counts()

print(counts.to_string())
print(f"\nEmbargo aplicado: {EMBARGO} pasos = {EMBARGO * step}")
for name in ("train", "valid", "test"):
    window = engineered.index[split == name]
    print(f"{name:6s} {window.min()} -> {window.max()}  ({len(window):,} filas)")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.2))
colours = {
    "train": COLOR_NAMES["cornflower blue"],
    "valid": COLOR_NAMES["ochre-gold"],
    "test": COLOR_NAMES["crimson"],
    "unused": "#e5e7eb",
}
for name, colour in colours.items():
    mask = (split == name).to_numpy()
    ax.fill_between(engineered.index, 0, mask.astype(float), color=colour, label=name, step="mid")
ax.set_yticks([])
ax.set_title("Partición temporal, con embargo entre bloques")
ax.legend(ncols=4, loc="upper center", bbox_to_anchor=(0.5, -0.15))
savefig(fig, f"{FIG}/splits.png")

## 4. Imputación y escalado

Aquí es donde importa el orden. El `Pipeline` se ajusta **sólo con train**, y
después se aplica a las tres particiones. `ColumnTransformer` no es adorno
burocrático: es lo que hace que el mismo objeto que se ajusta aquí pueda
serializarse y reaplicarse en `04` y en `05` sin reimplementar nada, que es la
única forma de garantizar que las tres etapas hacen exactamente la misma
transformación.

Antes del imputador se interpolan los huecos cortos, porque una interpolación
temporal usa la vecindad y conserva la forma de la serie, mientras que la
mediana la aplana. Los huecos largos llegan al imputador y, si aún quedan filas
sin objetivo, se caen: una fila sin `y` no enseña nada y una `y` imputada es una
etiqueta inventada.

In [ ]:
# La interpolación temporal es local y no usa estadísticos globales, así que
# aplicarla antes de partir no filtra información entre particiones.
interpolated = engineered.interpolate(method="time", limit=MAX_INTERPOLATION, limit_area="inside")

# Una y imputada es una etiqueta inventada: esas filas se van.
labelled = interpolated[interpolated[TARGET].notna()]
split = split.loc[labelled.index]

feature_names = [name for name in labelled.columns if name != TARGET and name != GROUP]
print(f"{len(labelled):,} filas con objetivo observado, {len(feature_names)} predictores")
print(f"Huecos restantes en los predictores: {int(labelled[feature_names].isna().sum().sum()):,}")

In [ ]:
scalers = {"standard": StandardScaler, "robust": RobustScaler}
preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            Pipeline(
                [
                    ("impute", SimpleImputer(strategy="median", add_indicator=True)),
                    ("scale", scalers[SCALER]()),
                ]
            ),
            feature_names,
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

train_rows = split == "train"
preprocessor.fit(labelled.loc[train_rows, feature_names])

# `add_indicator=True` añade una columna binaria por variable que tenía huecos.
# Vale la pena: "este dato faltaba" es a menudo informativo por sí mismo, y sin
# el indicador el modelo no puede distinguir un valor medido de uno imputado.
transformed_names = list(preprocessor.get_feature_names_out())
print(f"{len(feature_names)} -> {len(transformed_names)} columnas tras imputar y escalar")
[name for name in transformed_names if name.startswith("missingindicator")]

In [ ]:
matrix = pd.DataFrame(
    preprocessor.transform(labelled[feature_names]),
    index=labelled.index,
    columns=transformed_names,
)
matrix[TARGET] = labelled[TARGET]
matrix["split"] = split
if GROUP is not None:
    matrix[GROUP] = labelled[GROUP]

# Las primeras filas quedaron incompletas por los retardos y las ventanas
# móviles; una vez construido todo ya se sabe cuántas son.
matrix = matrix.dropna(subset=[TARGET])
print(matrix.shape)
matrix.head()

## 5. Comprobaciones

Cuatro afirmaciones que tienen que sostenerse antes de escribir nada. Son
`assert` y no comentarios porque un supuesto que no se comprueba es un supuesto
que un día deja de cumplirse sin avisar, y el síntoma aparece tres notebooks más
adelante disfrazado de resultado sospechosamente bueno.

In [ ]:
train_matrix = matrix.loc[matrix["split"] == "train", transformed_names]

# 1. El escalado se ajustó con train, así que es train quien debe quedar centrado.
#    Si valid o test lo estuvieran igual de bien, el escalador habría visto sus datos.
centred = train_matrix.mean().abs().max()
assert centred < 0.5, f"train no está centrado: |media| máxima = {centred:.3f}"

# 2. Ningún hueco sobrevive al imputador.
assert not matrix[transformed_names].isna().to_numpy().any(), "quedan NaN tras el pipeline"

# 3. El embargo separa las particiones en el tiempo.
bounds = matrix.groupby("split", observed=True).apply(
    lambda block: pd.Series({"inicio": block.index.min(), "fin": block.index.max()}),
    include_groups=False,
)
assert bounds.loc["train", "fin"] < bounds.loc["valid", "inicio"], "train y valid se solapan"
assert bounds.loc["valid", "fin"] < bounds.loc["test", "inicio"], "valid y test se solapan"

# 4. Ninguna columna quedó constante: el escalado de una columna sin varianza
#    produce ceros, y un predictor de ceros es ruido con nombre.
constant = train_matrix.columns[train_matrix.std() == 0].tolist()
assert not constant, f"columnas constantes en train: {constant}"

print("Las cuatro comprobaciones pasan.")
bounds

In [ ]:
# El desplazamiento de la media entre particiones es una medida de deriva. No es
# un error -- el clima cambia y las particiones son bloques temporales distintos --
# pero un salto grande anticipa que el modelo va a degradarse en test, y es mejor
# saberlo ahora que atribuirlo al modelo después.
drift = pd.DataFrame(
    {
        name: matrix.loc[matrix["split"] == name, transformed_names].mean()
        for name in ("train", "valid", "test")
    }
)
drift["deriva_test"] = (drift["test"] - drift["train"]).abs()
drift.sort_values("deriva_test", ascending=False).head(15).round(3)

## 6. Escritura

La matriz va a `gold` con la columna `split` incluida, de forma que `03`, `04` y
`05` heredan exactamente la misma partición sin volver a calcularla. Recalcularla
en cada notebook es la forma más silenciosa de comparar modelos evaluados sobre
conjuntos distintos.

El transformador va a `models/`. Es el objeto que hace reproducible todo lo
demás: sin él, la matriz es un fichero de números cuyo origen no se puede
reconstruir.

In [ ]:
destination = write_table(matrix.reset_index(), settings.paths.gold / OUTPUT)
artefact = settings.paths.models / "preprocessor.joblib"
joblib.dump(
    {
        "preprocessor": preprocessor,
        "feature_names": feature_names,
        "transformed_names": transformed_names,
        "target": TARGET,
        "step": step,
        "params": {
            "DROP": DROP,
            "LAGS": LAGS,
            "ROLLING": ROLLING,
            "CIRCULAR": CIRCULAR,
            "TRAIN_END": TRAIN_END,
            "VALID_END": VALID_END,
            "EMBARGO": EMBARGO,
            "MAX_INTERPOLATION": MAX_INTERPOLATION,
            "SCALER": SCALER,
        },
    },
    artefact,
)

write_table(drift.reset_index(names="variable"), TABLES / "summary.csv")
print(destination)
print(artefact)

## Siguiente paso

Cuando `LAGS`, `ROLLING` y `DROP` lleven un par de semanas sin cambiar, esto pasa
a `src/`:

1. Las funciones marcadas `# -> src/...` se mueven a
   `src/packagename/features/build.py` y `src/packagename/data/splits.py`, con
   sus tests en `tests/`.
2. Los parámetros de la celda de arriba pasan a una sección `preprocessing:` de
   `configs/config.yaml`, y a `PreprocessingSettings` en
   `src/packagename/config.py`.
3. Se registra una etapa en `src/packagename/etl/pipeline.py` que lea de
   `silver` y escriba en `gold`, después de las etapas que producen su entrada.
   Que los parámetros vivan en `configs/config.yaml` importa tanto como la
   propia etapa: cambiar `EMBARGO` no mueve la fecha de ningún fichero y sin
   embargo invalida todo lo que se derive de él — y en la config versionada ese
   cambio queda en `git log`.
4. El notebook se queda, reducido a llamar a la etapa y mirar las figuras.